# 14 — Step-Back Prompting RAG

> **Tier 3 | Query Handling**

## The Problem

Specific questions often miss relevant background context:

```
Specific Q: "Why did Arctic sea ice reach a record low in September 2023?"

Standard RAG: searches for "Arctic sea ice record low September 2023"
→ may retrieve only that specific event
→ misses the underlying physics: albedo feedback, ocean heat, jet stream changes
```

Without the conceptual background, the LLM's answer is shallow — it states the
fact but cannot explain the mechanism.

## Step-Back Prompting Solution

Reformulate the specific question into a broader **step-back question** that
retrieves the general principles, then combine both retrievals:

```
Step-back Q: "What are the physical mechanisms driving Arctic sea ice decline?"
→ retrieves passages about albedo feedback, polar amplification, ocean heat transport

Specific Q:  "Why did Arctic sea ice reach a record low in September 2023?"
→ retrieves passages about the specific event

Combined context → LLM answers with both background understanding + specific facts
```

## Flow Diagram

```mermaid
flowchart TD
    Q(["❓ Specific Question"])

    Q --> SBG["🧠 LLM Step-Back Generator\n→ Broader principle question"]
    Q --> QVEC["📐 Embed original question"]
    SBG --> SBVEC["📐 Embed step-back question"]

    subgraph RETRIEVAL ["🔍  Dual Retrieval"]
        direction LR
        R1["Specific retrieval\ntop-K chunks"]
        R2["Step-back retrieval\ntop-K chunks"]
    end

    QVEC  --> R1
    SBVEC --> R2

    R1 --> MERGE["🔀 Merge & Deduplicate"]
    R2 --> MERGE

    MERGE --> SYNTH["🟠 Strands Agent\nAnswer with background + specifics"]
    SYNTH --> ANS(["✅ Grounded, contextual answer"])

    style RETRIEVAL fill:#dbeafe,stroke:#3b82f6,color:#1e3a5f
    style MERGE     fill:#dcfce7,stroke:#16a34a,color:#14532d
    style SYNTH     fill:#ffedd5,stroke:#f97316,color:#7c2d12
```

In [ ]:
from IPython.display import display as _d, HTML as _H
_d(_H("""
<script src="https://cdn.jsdelivr.net/npm/mermaid@10/dist/mermaid.min.js"></script>
<div class="mermaid" style="background:#fff;padding:16px;border-radius:8px;">
flowchart TD
    Q(["❓ Specific Question"])

    Q --> SBG["🧠 LLM Step-Back Generator\n→ Broader principle question"]
    Q --> QVEC["📐 Embed original question"]
    SBG --> SBVEC["📐 Embed step-back question"]

    subgraph RETRIEVAL ["🔍  Dual Retrieval"]
        direction LR
        R1["Specific retrieval\ntop-K chunks"]
        R2["Step-back retrieval\ntop-K chunks"]
    end

    QVEC  --> R1
    SBVEC --> R2

    R1 --> MERGE["🔀 Merge & Deduplicate"]
    R2 --> MERGE

    MERGE --> SYNTH["🟠 Strands Agent\nAnswer with background + specifics"]
    SYNTH --> ANS(["✅ Grounded, contextual answer"])

    style RETRIEVAL fill:#dbeafe,stroke:#3b82f6,color:#1e3a5f
    style MERGE     fill:#dcfce7,stroke:#16a34a,color:#14532d
    style SYNTH     fill:#ffedd5,stroke:#f97316,color:#7c2d12
</div>
<script>mermaid.initialize({startOnLoad:true,theme:'default',flowchart:{curve:'basis'}});</script>
"""))

In [ ]:
import subprocess, sys
packages = ["boto3","qdrant-client","opensearch-py","requests-aws4auth",
            "strands-agents","pypdf","python-dotenv"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet"] + packages)
print("All packages ready.")

## Step 2 — Imports & Configuration

In [ ]:
import os, json, time, uuid, re
from typing import List, Dict, Optional

import boto3, pypdf
from dotenv import load_dotenv
from strands import Agent
from strands.models.bedrock import BedrockModel
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, VectorParams, PointStruct,
    Filter, FieldCondition, MatchValue
)

load_dotenv(r"C:\Users\Administrator\RAG\.env", override=True)

AWS_REGION      = os.getenv("AWS_DEFAULT_REGION", "us-east-1")
EMBEDDING_MODEL = "amazon.titan-embed-text-v2:0"
LLM_MODEL       = "us.anthropic.claude-sonnet-4-6"

QDRANT_URL     = os.getenv("QDRANT_URL", "")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY", "")
OPENSEARCH_URL = os.getenv("OPENSEARCH_ENDPOINT", "")

COLLECTION_NAME = "step_back_prompting_14"
EMBEDDING_DIM   = 1024
TOP_K           = 4    # hits per retrieval pass
CHUNK_SIZE      = 500
CHUNK_OVERLAP   = 50
PDF_PATH        = r"C:\Users\Administrator\RAG\data\climate.pdf"

print(f"Collection : {COLLECTION_NAME}")
print(f"PDF exists : {os.path.exists(PDF_PATH)}")
print(f"AWS region : {AWS_REGION}")
print(f"Key ID     : {os.getenv('AWS_ACCESS_KEY_ID','NOT SET')[:12]}...")

## Step 3 — Vector Store

In [1]:
class VectorStore:
    def __init__(self, collection_name, qdrant_url='', qdrant_api_key='',
                 opensearch_url='', region='us-east-1'):
        self.name = collection_name
        self._backend = None
        if qdrant_url:
            try:
                kw = {'url': qdrant_url}
                if qdrant_api_key: kw['api_key'] = qdrant_api_key
                self._qdrant = QdrantClient(**kw)
                self._qdrant.get_collections()
                self._backend = 'qdrant_cloud'
                print(f'Qdrant Cloud: {qdrant_url}')
                return
            except Exception as e:
                print(f'Qdrant Cloud unavailable ({e}), trying next...')
        if opensearch_url:
            try:
                from opensearchpy import OpenSearch, RequestsHttpConnection, AWSV4SignerAuth
                creds = boto3.Session().get_credentials()
                auth  = AWSV4SignerAuth(creds, region, 'aoss')
                host  = opensearch_url.replace('https://','').replace('http://','')
                self._os = OpenSearch(
                    hosts=[{'host': host, 'port': 443}],
                    http_auth=auth, use_ssl=True, verify_certs=True,
                    connection_class=RequestsHttpConnection, timeout=30)
                self._os.info()
                self._backend = 'opensearch'
                print(f'OpenSearch: {opensearch_url}')
                return
            except Exception as e:
                print(f'OpenSearch unavailable ({e}), falling back...')
        self._qdrant  = QdrantClient(':memory:')
        self._backend = 'qdrant_memory'
        print('Using Qdrant in-memory')

    def create_collection(self, dim=1024, force_recreate=False):
        if self._backend in ('qdrant_cloud', 'qdrant_memory'):
            exists = self.name in [c.name for c in self._qdrant.get_collections().collections]
            if exists and force_recreate:
                self._qdrant.delete_collection(self.name); exists = False
            if not exists:
                self._qdrant.create_collection(
                    self.name, vectors_config=VectorParams(size=dim, distance=Distance.COSINE))
                print(f'Created "{self.name}" (dim={dim})')
            else:
                print(f'"{self.name}" already exists — skipping recreate')

    def upsert(self, docs: List[Dict]) -> int:
        if self._backend in ('qdrant_cloud', 'qdrant_memory'):
            pts = [PointStruct(id=str(uuid.uuid4()), vector=d['embedding'],
                               payload={'text': d['text'], 'metadata': d.get('metadata', {})})
                   for d in docs]
            self._qdrant.upsert(collection_name=self.name, points=pts)
            return len(pts)

    def search(self, qvec: List[float], top_k: int = 5,
               query_filter=None) -> List[Dict]:
        if self._backend in ('qdrant_cloud', 'qdrant_memory'):
            resp = self._qdrant.query_points(
                collection_name=self.name, query=qvec, limit=top_k,
                query_filter=query_filter, with_payload=True)
            return [{'text': p.payload.get('text', ''),
                     'metadata': p.payload.get('metadata', {}),
                     'score': p.score, 'id': str(p.id)}
                    for p in resp.points]
        return []

    def count(self) -> int:
        if self._backend in ('qdrant_cloud', 'qdrant_memory'):
            return self._qdrant.get_collection(self.name).points_count or 0
        return 0

    def delete_collection(self):
        if self._backend in ('qdrant_cloud', 'qdrant_memory'):
            self._qdrant.delete_collection(self.name)
        print(f'Deleted "{self.name}"')

print("VectorStore defined.")

NameError: name 'List' is not defined

## Step 4 — Bedrock Helpers

In [ ]:
bedrock_rt = boto3.client('bedrock-runtime', region_name=AWS_REGION)

def embed_text(text: str) -> List[float]:
    body = json.dumps({"inputText": text, "dimensions": EMBEDDING_DIM, "normalize": True})
    resp = bedrock_rt.invoke_model(
        modelId=EMBEDDING_MODEL, body=body,
        contentType="application/json", accept="application/json")
    return json.loads(resp["body"].read())["embedding"]

def embed_batch(texts: List[str], label: str = '') -> List[List[float]]:
    out = []
    for i, t in enumerate(texts):
        out.append(embed_text(t))
        if (i + 1) % 20 == 0:
            print(f'  {label} {i+1}/{len(texts)}')
        time.sleep(0.04)
    return out

_model = BedrockModel(model_id=LLM_MODEL, region_name=AWS_REGION)

def llm(prompt: str) -> str:
    return str(Agent(model=_model, system_prompt="You are a helpful assistant.")(prompt))

test_emb = embed_text("step back prompting test")
print(f"Embedding OK — dim={len(test_emb)}")
print("Bedrock LLM ready.")

## Step 5 — Connect & Index climate.pdf

In [ ]:
def recursive_split(text: str, chunk_size: int, chunk_overlap: int) -> List[str]:
    separators = ["\n\n", "\n", ". ", " ", ""]
    def _split(text, seps):
        sep, new_seps = '', []
        for i, s in enumerate(seps):
            if s == '' or s in text:
                sep, new_seps = s, seps[i+1:]; break
        parts = text.split(sep) if sep != '' else list(text)
        good = []
        for part in parts:
            if len(part) <= chunk_size: good.append(part)
            elif new_seps: good.extend(_split(part, new_seps))
            else:
                for k in range(0, len(part), chunk_size - chunk_overlap):
                    good.append(part[k:k+chunk_size])
        chunks, cur_pieces, cur_len = [], [], 0
        for piece in good:
            p = piece.strip()
            if not p: continue
            addition = len(sep) + len(p) if cur_pieces else len(p)
            if cur_len + addition <= chunk_size:
                cur_pieces.append(p); cur_len += addition
            else:
                if cur_pieces: chunks.append(sep.join(cur_pieces))
                ov, ovl = [], 0
                for prev in reversed(cur_pieces):
                    if ovl + len(prev) + len(sep) <= chunk_overlap:
                        ov.insert(0, prev); ovl += len(prev) + len(sep)
                    else: break
                cur_pieces = ov + [p]
                cur_len = sum(len(x) + len(sep) for x in cur_pieces)
        if cur_pieces: chunks.append(sep.join(cur_pieces))
        return [c for c in chunks if c.strip()]
    return _split(text, separators)

vs = VectorStore(
    collection_name=COLLECTION_NAME,
    qdrant_url=QDRANT_URL,
    qdrant_api_key=QDRANT_API_KEY,
    opensearch_url=OPENSEARCH_URL,
    region=AWS_REGION
)
print(f"Backend: {vs._backend}")
vs.create_collection(dim=EMBEDDING_DIM, force_recreate=True)

reader    = pypdf.PdfReader(PDF_PATH)
full_text = ''
page_map  = []
for pg_idx, page in enumerate(reader.pages):
    pg_text = (page.extract_text() or '') + '\n\n'
    page_map.extend([pg_idx + 1] * len(pg_text))
    full_text += pg_text

chunks = recursive_split(full_text, CHUNK_SIZE, CHUNK_OVERLAP)
print(f"PDF pages: {len(reader.pages)}  |  Chunks: {len(chunks)}")

embs = embed_batch(chunks, label='[index]')
docs = []
for i, (chunk, emb) in enumerate(zip(chunks, embs)):
    char_offset = full_text.find(chunk[:40])
    page_num = page_map[min(char_offset, len(page_map)-1)] if char_offset >= 0 else 1
    docs.append({
        'text'     : chunk,
        'embedding': emb,
        'metadata' : {'chunk_idx': i, 'page_num': page_num, 'char_count': len(chunk)}
    })
vs.upsert(docs)
print(f"Indexed {vs.count()} vectors into '{COLLECTION_NAME}'")

## Step 6 — Step-Back Question Generator

Given a specific question, the LLM produces one broader question that covers
the underlying principle or concept needed to answer the original well.

In [ ]:
def generate_step_back(question: str) -> str:
    prompt = f"""You are an expert at reformulating questions.
Given a specific question, produce ONE broader "step-back" question that asks about
the general principle, concept, or background knowledge needed to answer the original.

Rules:
- The step-back question must be broader and more general than the original
- It should retrieve background/conceptual knowledge, not the specific answer
- Return ONLY the step-back question — no explanation, no quotes, no punctuation changes

Original question: {question}

Step-back question:"""
    raw = llm(prompt).strip()
    # Strip quotes if model wraps the answer
    raw = raw.strip('"').strip("'")
    return raw

# Examples
examples = [
    "What was the global mean surface temperature anomaly in 2023?",
    "How much has Arctic sea ice declined since 1979?",
    "What are the projected sea level rise figures for 2100 under SSP5-8.5?",
]

print("Step-Back Question Generation Examples")
print("=" * 65)
for q in examples:
    sb = generate_step_back(q)
    print(f"\nOriginal   : {q}")
    print(f"Step-back  : {sb}")

## Step 7 — Step-Back RAG Pipeline

Two retrieval passes (specific + step-back), deduplicated, then synthesised.

In [ ]:
def rag_step_back(question: str, top_k: int = TOP_K, verbose: bool = True) -> Dict:
    t0 = time.time()

    # 1. Generate step-back question
    step_back_q = generate_step_back(question)

    # 2. Retrieve for specific question
    specific_vec  = embed_text(question)
    specific_hits = vs.search(specific_vec, top_k=top_k)

    # 3. Retrieve for step-back question
    step_back_vec  = embed_text(step_back_q)
    step_back_hits = vs.search(step_back_vec, top_k=top_k)

    # 4. Merge and deduplicate by chunk id
    seen_ids  = set()
    merged    = []
    # Step-back hits first — background context comes first in the prompt
    for h in step_back_hits + specific_hits:
        if h['id'] not in seen_ids:
            seen_ids.add(h['id'])
            merged.append(h)

    # 5. Build context with source labels
    context_parts = []
    for h in merged:
        m   = h['metadata']
        src = f"[p.{m.get('page_num','?')}]"
        context_parts.append(f"{src}\n{h['text']}")
    context = '\n\n'.join(context_parts)

    # 6. Synthesise
    synth_prompt = (
        "You have two kinds of context below:\n"
        "  • Background passages (from the step-back question) — general principles\n"
        "  • Specific passages (from the original question) — direct evidence\n\n"
        "Use BOTH to give a thorough, well-grounded answer. "
        "Cite page numbers (e.g. [p.3]) for key facts.\n\n"
        f"Context:\n{context}\n\n"
        f"Original question: {question}\n\nAnswer:"
    )
    answer  = llm(synth_prompt)
    latency = (time.time() - t0) * 1000

    if verbose:
        print(f"\nOriginal Q : {question}")
        print(f"Step-back Q: {step_back_q}")
        print(f"\nSpecific hits  ({len(specific_hits)}): "
              f"pages {sorted({h['metadata'].get('page_num','?') for h in specific_hits})}")
        print(f"Step-back hits ({len(step_back_hits)}): "
              f"pages {sorted({h['metadata'].get('page_num','?') for h in step_back_hits})}")
        print(f"Unique merged  : {len(merged)} passages  |  Latency: {latency:.0f}ms")
        print(f"\nAnswer:\n{answer}")
        print('-' * 70)

    return {
        'question'      : question,
        'step_back_q'   : step_back_q,
        'specific_hits' : specific_hits,
        'step_back_hits': step_back_hits,
        'merged_hits'   : merged,
        'answer'        : answer,
        'latency_ms'    : latency,
    }

# Demo
rag_step_back("How much has Arctic sea ice declined since 1979?")

## Step 8 — Baseline Comparison: Simple vs Step-Back

Show side-by-side which pages each strategy retrieves and whether the
step-back pass adds new background pages not found by the specific query.

In [ ]:
def rag_simple(question: str, top_k: int = TOP_K) -> Dict:
    qvec = embed_text(question)
    hits = vs.search(qvec, top_k=top_k)
    context = '\n\n'.join(f"[p.{h['metadata'].get('page_num','?')}]\n{h['text']}" for h in hits)
    answer  = llm(
        f"Answer using ONLY the context. Cite page numbers.\n\nContext:\n{context}\n\nQ: {question}\n\nA:"
    )
    return {'hits': hits, 'answer': answer}

test_questions = [
    "What are the projected sea level rise figures for 2100?",
    "What role do carbon sinks play in the global carbon cycle?",
    "How have extreme weather events changed in frequency and intensity?",
]

print(f"{'Question':<52}  {'Simple pages':>14}  {'Step-back pages':>16}  {'New pages':>10}")
print('-' * 98)

for q in test_questions:
    r_simple    = rag_simple(q)
    r_sb        = rag_step_back(q, verbose=False)

    simple_pages   = set(h['metadata'].get('page_num','?') for h in r_simple['hits'])
    specific_pages = set(h['metadata'].get('page_num','?') for h in r_sb['specific_hits'])
    sb_pages       = set(h['metadata'].get('page_num','?') for h in r_sb['step_back_hits'])
    new_pages      = sb_pages - specific_pages

    print(f"{q[:51]:<52}  {str(sorted(simple_pages)):>14}  "
          f"{str(sorted(sb_pages)):>16}  {str(sorted(new_pages)):>10}")

## Step 9 — Full Step-Back Answer Demo

Show the complete answer for a highly specific question where background
context makes a meaningful difference.

In [ ]:
rag_step_back(
    "What are the consequences of permafrost thaw for global carbon budgets?",
    top_k=4
)

## Step 10 — Summary

| Component | Implementation |
|-----------|---------------|
| Step-back generator | Claude Sonnet 4.6 → one broader principle question |
| Retrieval pass 1 | Original specific question → top-K chunks |
| Retrieval pass 2 | Step-back question → top-K chunks |
| Merge | Deduplicate by chunk ID; background context placed first |
| Synthesis | Single LLM call; prompt labels background vs specific passages |
| Vector DB | Qdrant Cloud → OpenSearch → in-memory |
| Embeddings | Amazon Bedrock Titan V2 (1024-dim) |

## How Step-Back differs from Query Decomposition (nb 13)

| | Query Decomposition | Step-Back Prompting |
|---|---|---|
| Direction | Breaks one Q into N sub-Qs (horizontal split) | Zooms out from specific to general (vertical) |
| Use case | Multi-part questions | Specific questions needing background context |
| LLM calls | 1 decompose + 1 synthesise | 1 step-back + 1 synthesise |
| Retrieval passes | N (one per sub-Q) | 2 (specific + step-back) |

## When to use Step-Back

- Question asks about a specific fact that requires conceptual background to explain
- User is new to the domain — needs the "why" not just the "what"
- Factual questions where the document has principles spread across different pages
- Any question containing *why*, *how does it work*, *what causes*, *what explains*

### Next: **15 — Fusion Retrieval**

In [ ]:
# vs.delete_collection()  # uncomment to clean up
print(f"Collection '{COLLECTION_NAME}' in {vs._backend} — {vs.count()} vectors")
print("\nDone. Give the go-ahead for notebook 15.")